AI Assistance: OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization, and code methodological brainstorming. All final modeling, implementation, validation, commentary, and interpretation were performed and verified by the authors.

In [1]:
import pandas as pd
import duckdb
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path

platform = 'deepnote'

if platform == 'deepnote':
    PROCESSED_DIR = Path('data/Processed')
if platform == 'vscode':
    PROCESSED_DIR = Path('work/Processed')

# I\. Load train set

In [2]:
df_train = pd.read_parquet(PROCESSED_DIR / 'omni_train_cleaned_log.parquet')

,datetime,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,...,day_sin,hour_cos,hour_sin,minute_cos,minute_sin,kp_10,data_split,mag_avg_nt_log,flow_speed_km_s_log,proton_density_n_cc_log
0,2000-01-01 00:00:00,2000,1,0,0,6.820,-5.940,0.27,-0.08,664.7,...,0.017213,1.0,0.0,1.000000,0.000000,53,train,2.056685,6.500839,1.415853
1,2000-01-01 00:01:00,2000,1,0,1,6.990,-5.880,1.95,1.08,664.7,...,0.017213,1.0,0.0,0.994522,0.104528,53,train,2.078191,6.500839,1.415853
2,2000-01-01 00:02:00,2000,1,0,2,6.990,-5.710,2.74,2.24,663.2,...,0.017213,1.0,0.0,0.978148,0.207912,53,train,2.078191,6.498583,1.444563
3,2000-01-01 00:03:00,2000,1,0,3,6.830,-5.330,3.18,2.78,662.2,...,0.017213,1.0,0.0,0.951057,0.309017,53,train,2.057963,6.497077,1.413423
4,2000-01-01 00:04:00,2000,1,0,4,6.905,-4.565,2.99,3.76,675.3,...,0.017213,1.0,0.0,0.913545,0.406737,53,train,2.067495,6.516637,1.342865


In [3]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 10334497 entries, 0 to 10334496
Data columns (total 22 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   datetime                 datetime64[us]
 1   year                     int64         
 2   day                      int64         
 3   hour                     int64         
 4   minute                   int64         
 5   mag_avg_nt               float64       
 6   bx_gsm_nt                float64       
 7   by_gsm_nt                float64       
 8   bz_gsm_nt                float64       
 9   flow_speed_km_s          float64       
 10  proton_density_n_cc      float64       
 11  day_cos                  float64       
 12  day_sin                  float64       
 13  hour_cos                 float64       
 14  hour_sin                 float64       
 15  minute_cos               float64       
 16  minute_sin               float64       
 17  kp_10                    int64      

In [5]:
# Confirm no nulls survived cleaning
print(df_train.isna().sum())

datetime                   0
year                       0
day                        0
hour                       0
minute                     0
mag_avg_nt                 0
bx_gsm_nt                  0
by_gsm_nt                  0
bz_gsm_nt                  0
flow_speed_km_s            0
proton_density_n_cc        0
day_cos                    0
day_sin                    0
hour_cos                   0
hour_sin                   0
minute_cos                 0
minute_sin                 0
kp_10                      0
data_split                 0
mag_avg_nt_log             0
flow_speed_km_s_log        0
proton_density_n_cc_log    0
dtype: int64


In [6]:
# Check for duplicate minutes within the same year/day/hour in df_train
duplicate_minutes = df_train[
    df_train.duplicated(subset=['year', 'day', 'hour', 'minute'], keep=False)
]

print(f'Found {len(duplicate_minutes):,} rows with duplicate (year, day, hour, minute) combinations')
duplicate_minutes.sort_values(['year', 'day', 'hour', 'minute']).head(20)

Found 0 rows with duplicate (year, day, hour, minute) combinations


,datetime,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,...,day_sin,hour_cos,hour_sin,minute_cos,minute_sin,kp_10,data_split,mag_avg_nt_log,flow_speed_km_s_log,proton_density_n_cc_log


# II\. Create lag features \- train set

In [7]:
# Create the lagged features and drop intervals missing >5 minutes, including the lookback periods

KP_FEATURE_VARS = [
    'mag_avg_nt_log',
    'bx_gsm_nt',
    'by_gsm_nt',
    'bz_gsm_nt',
    'flow_speed_km_s_log',
    'proton_density_n_cc_log',
]

KP_LOOKBACK_WINDOWS = [('0_1h', 1), ('1_2h', 2), ('2_3h', 3), ('3_4h', 4)]


def build_kp_interval_features(df, value_cols=KP_FEATURE_VARS, max_missing_minutes=5):
    print(f'--- Record count audit ---')
    print(f'Input rows (per-minute): {len(df):,}')

    d = df.set_index('datetime').sort_index()

    minute_counts = d.resample('1h').size()
    hourly_stats = d[value_cols].resample('1h').agg(['mean', 'min', 'max'])
    hourly_stats.columns = [f'{col}_{stat}' for col, stat in hourly_stats.columns]
    kp_hourly_mean = d['kp_10'].resample('1h').mean()

    idx = hourly_stats.index
    print(f'Resampled hourly buckets ({idx.min()} to {idx.max()}): {len(idx):,}')

    missing = 60 - minute_counts

    # shifts past the edge of the resampled range have no underlying data, so treat them as fully missing
    window_missing = {label: missing.shift(k).fillna(60) for label, k in KP_LOOKBACK_WINDOWS}
    block_missing = missing + missing.shift(-1).fillna(60) + missing.shift(-2).fillna(60)

    interval_index = idx[idx.hour % 3 == 0]
    print(f'Candidate 3-hour intervals (hour % 3 == 0): {len(interval_index):,}')

    features = pd.DataFrame(index=interval_index)
    features.index.name = 'datetime'
    # Kp is only reported once per 3-hour block (the same value repeats across all 3 hours),
    # so the interval's own hourly value is the block value - no averaging across hours needed.
    features['kp_10'] = kp_hourly_mean.loc[interval_index]
    features['kp_index'] = (features['kp_10'] / 10.0).round(2)
    features['year'] = interval_index.year
    features['day'] = interval_index.dayofyear
    features['hour'] = interval_index.hour
    features['minute'] = 0
    features['day_cos'] = np.cos(2 * np.pi * features['day'] / 365)
    features['day_sin'] = np.sin(2 * np.pi * features['day'] / 365)
    features['hour_cos'] = np.cos(2 * np.pi * features['hour'] / 24)
    features['hour_sin'] = np.sin(2 * np.pi * features['hour'] / 24)
    features['minute_cos'] = 1.0
    features['minute_sin'] = 0.0
    features['data_split'] = df['data_split'].iloc[0]

    for label, shift_by in KP_LOOKBACK_WINDOWS:
        for var in value_cols:
            features[f'{var}_avg_{label}'] = hourly_stats[f'{var}_mean'].shift(shift_by).loc[interval_index]
            features[f'{var}_min_{label}'] = hourly_stats[f'{var}_min'].shift(shift_by).loc[interval_index]
            features[f'{var}_max_{label}'] = hourly_stats[f'{var}_max'].shift(shift_by).loc[interval_index]

    print(f'Assembled feature rows (before drop filtering): {len(features):,}, columns: {features.shape[1]:,}')

    fail_block = block_missing.loc[interval_index] > max_missing_minutes
    fail_windows = {
        label: window_missing[label].loc[interval_index] > max_missing_minutes
        for label, _ in KP_LOOKBACK_WINDOWS
    }
    drop_mask = fail_block.copy()
    for label in fail_windows:
        drop_mask |= fail_windows[label]

    reasons = pd.DataFrame({'block_3h': fail_block}, index=interval_index)
    for label in fail_windows:
        reasons[label] = fail_windows[label]
    dropped_reasons = reasons.loc[drop_mask]

    print(f'Evaluated {len(interval_index):,} candidate 3-hour Kp intervals '
          f'({interval_index.min()} to {interval_index.max()})')
    print(f'Dropping {drop_mask.sum():,} intervals ({drop_mask.mean() * 100:.2f}%) '
          f'for more than {max_missing_minutes} missing minutes in the 3-hour interval '
          f'or a lookback window:')
    print(f'  - 3-hour interval itself: {fail_block.sum():,}')
    for label, _ in KP_LOOKBACK_WINDOWS:
        print(f'  - {label} lookback window: {fail_windows[label].sum():,}')
    print(f'Keeping {(~drop_mask).sum():,} intervals')

    if drop_mask.any():
        print('\nSample of dropped intervals and failing checks:')
        print(dropped_reasons.head(10))
        if len(dropped_reasons) > 10:
            print('...')
            print(dropped_reasons.tail(5))

    kept = features.loc[~drop_mask.values].copy()

    print(f'\n--- Record count audit summary ---')
    print(f'{"Input per-minute rows:":<35}{len(df):>12,}')
    print(f'{"Resampled hourly buckets:":<35}{len(idx):>12,}')
    print(f'{"Candidate 3-hour intervals:":<35}{len(interval_index):>12,}')
    print(f'{"Dropped intervals:":<35}{drop_mask.sum():>12,}')
    print(f'{"Final kept intervals:":<35}{len(kept):>12,}')
    print(f'{"Final output shape:":<35}{str(kept.shape):>12}')
    assert len(kept) == len(interval_index) - drop_mask.sum(), 'kept count does not reconcile with candidates - drops'

    return kept, dropped_reasons

In [8]:
# Train set - Build time lag features and drop incomplete intervals

kp_features_train, kp_dropped_train = build_kp_interval_features(df_train)
kp_features_train.head()

--- Record count audit ---
Input rows (per-minute): 10,334,497
Resampled hourly buckets (2000-01-01 00:00:00 to 2019-12-31 23:00:00): 175,320
Candidate 3-hour intervals (hour % 3 == 0): 58,440
Assembled feature rows (before drop filtering): 58,440, columns: 85
Evaluated 58,440 candidate 3-hour Kp intervals (2000-01-01 00:00:00 to 2019-12-31 21:00:00)
Dropping 3,206 intervals (5.49%) for more than 5 missing minutes in the 3-hour interval or a lookback window:
  - 3-hour interval itself: 1,986
  - 0_1h lookback window: 1,250
  - 1_2h lookback window: 1,382
  - 2_3h lookback window: 1,231
  - 3_4h lookback window: 1,251
Keeping 55,234 intervals

Sample of dropped intervals and failing checks:
                     block_3h   0_1h   1_2h   2_3h   3_4h
datetime                                                 
2000-01-01 00:00:00     False   True   True   True   True
2000-01-01 03:00:00     False  False  False  False   True
2000-01-17 12:00:00      True  False  False  False  False
2000-01-17 

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,1.000000e+00,...,7.51,-2.650083,-7.15,4.43,6.569232,6.517967,6.651443,1.155989,0.978326,1.272566
2000-01-01 09:00:00,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,7.071068e-01,...,5.37,-0.691833,-4.90,4.35,6.577484,6.537126,6.650408,1.081418,1.011601,1.208960
2000-01-01 12:00:00,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,1.224647e-16,...,6.39,-0.399500,-6.13,6.58,6.619954,6.574378,6.654925,1.046978,0.896088,1.220830
2000-01-01 15:00:00,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,6.589432,6.557204,6.640529,1.133350,1.018847,1.223775
2000-01-01 18:00:00,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,-1.000000e+00,...,5.81,-0.374333,-2.88,4.80,6.564192,6.527812,6.629759,1.387579,0.989541,1.658228


In [9]:
kp_features_train.shape

(55234, 85)

# III\. Load validation set

In [10]:
df_validation = pd.read_parquet(PROCESSED_DIR / 'omni_validation_cleaned_log.parquet')

,datetime,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,...,day_sin,hour_cos,hour_sin,minute_cos,minute_sin,kp_10,data_split,mag_avg_nt_log,flow_speed_km_s_log,proton_density_n_cc_log
10334497,2020-01-01 00:00:00,2020,1,0,0,4.65,-3.30,3.20,-0.70,295.3,...,0.017213,1.0,0.0,1.000000,0.000000,3,validation,1.731656,5.691372,1.827770
10334498,2020-01-01 00:01:00,2020,1,0,1,4.67,-3.29,3.23,-0.77,295.3,...,0.017213,1.0,0.0,0.994522,0.104528,3,validation,1.735189,5.691372,1.827770
10334499,2020-01-01 00:02:00,2020,1,0,2,3.84,-1.44,2.25,0.09,295.6,...,0.017213,1.0,0.0,0.978148,0.207912,3,validation,1.576915,5.692384,1.842928
10334500,2020-01-01 00:03:00,2020,1,0,3,4.64,-3.31,3.22,-0.49,295.9,...,0.017213,1.0,0.0,0.951057,0.309017,3,validation,1.729884,5.693395,1.857859
10334501,2020-01-01 00:04:00,2020,1,0,4,4.63,-3.29,3.19,-0.69,295.9,...,0.017213,1.0,0.0,0.913545,0.406737,3,validation,1.728109,5.693395,1.857859


In [11]:
df_validation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1800962 entries, 10334497 to 12135458
Data columns (total 22 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   datetime                 datetime64[us]
 1   year                     int64         
 2   day                      int64         
 3   hour                     int64         
 4   minute                   int64         
 5   mag_avg_nt               float64       
 6   bx_gsm_nt                float64       
 7   by_gsm_nt                float64       
 8   bz_gsm_nt                float64       
 9   flow_speed_km_s          float64       
 10  proton_density_n_cc      float64       
 11  day_cos                  float64       
 12  day_sin                  float64       
 13  hour_cos                 float64       
 14  hour_sin                 float64       
 15  minute_cos               float64       
 16  minute_sin               float64       
 17  kp_10                    int64

In [13]:
# Confirm no nulls survived cleaning
print(df_validation.isna().sum())

datetime                   0
year                       0
day                        0
hour                       0
minute                     0
mag_avg_nt                 0
bx_gsm_nt                  0
by_gsm_nt                  0
bz_gsm_nt                  0
flow_speed_km_s            0
proton_density_n_cc        0
day_cos                    0
day_sin                    0
hour_cos                   0
hour_sin                   0
minute_cos                 0
minute_sin                 0
kp_10                      0
data_split                 0
mag_avg_nt_log             0
flow_speed_km_s_log        0
proton_density_n_cc_log    0
dtype: int64


# IV\. Create lag features \- validation set

In [14]:
# Validation set - Build time lag features and drop incomplete intervals

kp_features_validation, kp_dropped_validation = build_kp_interval_features(df_validation)
kp_features_validation.head()

--- Record count audit ---
Input rows (per-minute): 1,800,962
Resampled hourly buckets (2020-01-01 00:00:00 to 2023-06-30 23:00:00): 30,648
Candidate 3-hour intervals (hour % 3 == 0): 10,216
Assembled feature rows (before drop filtering): 10,216, columns: 85
Evaluated 10,216 candidate 3-hour Kp intervals (2020-01-01 00:00:00 to 2023-06-30 21:00:00)
Dropping 503 intervals (4.92%) for more than 5 missing minutes in the 3-hour interval or a lookback window:
  - 3-hour interval itself: 334
  - 0_1h lookback window: 245
  - 1_2h lookback window: 247
  - 2_3h lookback window: 250
  - 3_4h lookback window: 246
Keeping 9,713 intervals

Sample of dropped intervals and failing checks:
                     block_3h   0_1h   1_2h   2_3h   3_4h
datetime                                                 
2020-01-01 00:00:00     False   True   True   True   True
2020-01-01 03:00:00     False  False  False  False   True
2020-01-06 12:00:00      True  False  False  False  False
2020-01-06 15:00:00     Fa

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-01 06:00:00,0.0,0.0,2020,1,6,0,0.999852,0.017213,6.123234e-17,1.000000e+00,...,-0.31,1.166000,0.52,1.83,5.715800,5.700444,5.726522,2.273935,1.986504,2.534094
2020-01-01 09:00:00,7.0,0.7,2020,1,9,0,0.999852,0.017213,-7.071068e-01,7.071068e-01,...,-0.92,1.366417,0.93,2.22,5.740880,5.717028,5.749711,2.253194,2.131797,2.378620
2020-01-01 12:00:00,7.0,0.7,2020,1,12,0,0.999852,0.017213,-1.000000e+00,1.224647e-16,...,-1.43,-3.002500,-3.44,-2.58,5.799468,5.774552,5.833933,2.208855,2.085672,2.397895
2020-01-01 15:00:00,13.0,1.3,2020,1,15,0,0.999852,0.017213,-7.071068e-01,-7.071068e-01,...,0.97,1.558583,-0.94,2.97,5.781606,5.771441,5.790572,2.354791,2.230014,2.483239
2020-01-01 18:00:00,10.0,1.0,2020,1,18,0,0.999852,0.017213,-1.836970e-16,-1.000000e+00,...,0.86,-0.565028,-2.66,2.10,5.804606,5.766131,5.869297,2.326269,2.239645,2.392426


In [15]:
kp_features_validation.shape

(9713, 85)

# V\. Load test set

In [16]:
df_test = pd.read_parquet(PROCESSED_DIR / 'omni_test_cleaned_log.parquet')

,datetime,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,...,day_sin,hour_cos,hour_sin,minute_cos,minute_sin,kp_10,data_split,mag_avg_nt_log,flow_speed_km_s_log,proton_density_n_cc_log
12135459,2023-06-30 23:59:00,2023,181,23,59,6.94,-3.93,4.75,2.79,515.00,...,0.025818,0.965926,-0.258819,0.994522,-0.104528,0,test,2.071913,6.246107,0.928219
12135460,2023-07-01 00:00:00,2023,182,0,0,7.04,-3.64,4.16,4.27,515.00,...,0.008607,1.000000,0.000000,1.000000,0.000000,20,test,2.084429,6.246107,0.928219
12135461,2023-07-01 00:01:00,2023,182,0,1,7.03,-3.76,4.18,4.17,503.25,...,0.008607,1.000000,0.000000,0.994522,0.104528,20,test,2.083185,6.223072,1.136229
12135462,2023-07-01 00:02:00,2023,182,0,2,7.06,-3.88,4.25,4.02,491.50,...,0.008607,1.000000,0.000000,0.978148,0.207912,20,test,2.086914,6.199494,1.308333
12135463,2023-07-01 00:03:00,2023,182,0,3,6.38,-2.79,4.11,3.72,479.75,...,0.008607,1.000000,0.000000,0.951057,0.309017,20,test,1.998774,6.175347,1.455121


In [17]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 1519035 entries, 12135459 to 13654493
Data columns (total 22 columns):
 #   Column                   Non-Null Count    Dtype         
---  ------                   --------------    -----         
 0   datetime                 1519035 non-null  datetime64[us]
 1   year                     1519035 non-null  int64         
 2   day                      1519035 non-null  int64         
 3   hour                     1519035 non-null  int64         
 4   minute                   1519035 non-null  int64         
 5   mag_avg_nt               1519035 non-null  float64       
 6   bx_gsm_nt                1519035 non-null  float64       
 7   by_gsm_nt                1519035 non-null  float64       
 8   bz_gsm_nt                1519035 non-null  float64       
 9   flow_speed_km_s          1519035 non-null  float64       
 10  proton_density_n_cc      1519035 non-null  float64       
 11  day_cos                  1519035 non-null  float64       
 12  day

In [20]:
# Confirm no nulls survived cleaning
print(df_test.isna().sum())

datetime                   0
year                       0
day                        0
hour                       0
minute                     0
mag_avg_nt                 0
bx_gsm_nt                  0
by_gsm_nt                  0
bz_gsm_nt                  0
flow_speed_km_s            0
proton_density_n_cc        0
day_cos                    0
day_sin                    0
hour_cos                   0
hour_sin                   0
minute_cos                 0
minute_sin                 0
kp_10                      0
data_split                 0
mag_avg_nt_log             0
flow_speed_km_s_log        0
proton_density_n_cc_log    0
dtype: int64


# VI\. Create lag features \- test set

In [21]:
# Test set - Build time lag features and drop incomplete intervals

kp_features_test, kp_dropped_test = build_kp_interval_features(df_test)
kp_features_test.head()

--- Record count audit ---
Input rows (per-minute): 1,519,035
Resampled hourly buckets (2023-06-30 23:00:00 to 2026-06-30 23:00:00): 26,305
Candidate 3-hour intervals (hour % 3 == 0): 8,768
Assembled feature rows (before drop filtering): 8,768, columns: 85
Evaluated 8,768 candidate 3-hour Kp intervals (2023-07-01 00:00:00 to 2026-06-30 21:00:00)
Dropping 610 intervals (6.96%) for more than 5 missing minutes in the 3-hour interval or a lookback window:
  - 3-hour interval itself: 451
  - 0_1h lookback window: 361
  - 1_2h lookback window: 380
  - 2_3h lookback window: 361
  - 3_4h lookback window: 362
Keeping 8,158 intervals

Sample of dropped intervals and failing checks:
                     block_3h   0_1h   1_2h   2_3h   3_4h
datetime                                                 
2023-07-01 00:00:00     False   True   True   True   True
2023-07-01 03:00:00     False  False  False  False   True
2023-07-01 06:00:00      True  False  False  False  False
2023-07-01 09:00:00      True

,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,hour_sin,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-07-01 18:00:00,10.0,1.0,2023,182,18,0,-0.999963,0.008607,-1.836970e-16,-1.000000,...,5.95,-1.277393,-1.78,-0.89,6.190540,6.179137,6.241445,1.098754,0.896088,1.141033
2023-07-01 21:00:00,13.0,1.3,2023,182,21,0,-0.999963,0.008607,7.071068e-01,-0.707107,...,3.46,0.402750,-0.74,1.55,6.095282,6.076495,6.110802,1.000582,0.928219,1.098612
2023-07-02 00:00:00,7.0,0.7,2023,183,0,0,-0.999963,-0.008607,1.000000e+00,0.000000,...,-0.42,1.983583,1.08,2.86,6.072620,6.042870,6.104347,1.303858,1.156881,1.456287
2023-07-02 03:00:00,13.0,1.3,2023,183,3,0,-0.999963,-0.008607,7.071068e-01,0.707107,...,4.75,1.654667,0.11,2.84,6.059389,6.028279,6.094472,1.508568,1.378766,1.591274
2023-07-02 06:00:00,7.0,0.7,2023,183,6,0,-0.999963,-0.008607,6.123234e-17,1.000000,...,5.15,-0.022833,-2.05,0.83,6.071296,6.041920,6.082219,1.417285,1.311032,1.543298


In [22]:
kp_features_test.shape

(8158, 85)

In [76]:
# Validate the record counts pre- and post-cleaning

# Numbers sourced from 'train_test_split_before_cleaning' notebook
raw_count = 13936320
train_count = 10519200
test_count = 1578241
validation_count = 1838879 

# Numbers sourced from '03_04_cleaning_eda'
train_dropped_count = 184703
test_dropped_count = 59206
val_dropped_count = 37917

train_cleaned_count = len(df_train)
test_cleaned_count = len(df_test)
val_cleaned_count = len(df_validation)

# Assert statements
assert raw_count == train_count + test_count + validation_count
assert train_count - train_dropped_count == train_cleaned_count

assert validation_count - val_dropped_count == val_cleaned_count
assert test_count - test_dropped_count == test_cleaned_count

# VII\. Final export to parquet

In [77]:
# Write engineered Kp interval features to parquet
train_output_path = PROCESSED_DIR / 'lr_features_0h_train.parquet'
validation_output_path = PROCESSED_DIR / 'lr_features_0h_validation.parquet'
test_output_path = PROCESSED_DIR / 'lr_features_0h_test.parquet'

kp_features_train.to_parquet(train_output_path)
kp_features_validation.to_parquet(validation_output_path)
kp_features_test.to_parquet(test_output_path)

print(f'Wrote {len(kp_features_train):,} rows to {train_output_path.as_posix()}')
print(f'Wrote {len(kp_features_validation):,} rows to {validation_output_path.as_posix()}')
print(f'Wrote {len(kp_features_test):,} rows to {test_output_path.as_posix()}')

Wrote 55,234 rows to work/Processed/lr_features_0h_train.parquet
Wrote 9,713 rows to work/Processed/lr_features_0h_validation.parquet
Wrote 8,158 rows to work/Processed/lr_features_0h_test.parquet


# VIII\. Create forecast targets

In [42]:
VALID_FORECAST_HORIZONS = {3, 6, 9, 12, 15, 18, 21}

def create_forecast_target(df, target_column='kp_index', forecast_horizon=3):
    if forecast_horizon not in VALID_FORECAST_HORIZONS:
        raise ValueError(
            f"forecast_horizon must be one of {sorted(VALID_FORECAST_HORIZONS)}, got {forecast_horizon}"
        )

    # Rows are already 3-hour intervals, so each 3hr step in the horizon shifts one row forward
    df = df.copy()
    row_shift = forecast_horizon // 3
    new_column = f"{target_column}_{forecast_horizon}_hr_forecast"
    df[new_column] = df[target_column].shift(-row_shift)
    return df[[new_column] + [col for col in df.columns if col != new_column]]

## 3\-hour forecast

In [43]:
# Create 3-hour forecast dataframes (train/validation/test)
kp_features_train_3hr = create_forecast_target(df=kp_features_train)
kp_features_validation_3hr = create_forecast_target(df=kp_features_validation)
kp_features_test_3hr = create_forecast_target(df=kp_features_test)

In [44]:
kp_features_train_3hr.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 55234 entries, 2000-01-01 06:00:00 to 2019-12-31 21:00:00
Data columns (total 86 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   kp_index_3_hr_forecast            55233 non-null  float64
 1   kp_10                             55234 non-null  float64
 2   kp_index                          55234 non-null  float64
 3   year                              55234 non-null  int32  
 4   day                               55234 non-null  int32  
 5   hour                              55234 non-null  int32  
 6   minute                            55234 non-null  int64  
 7   day_cos                           55234 non-null  float64
 8   day_sin                           55234 non-null  float64
 9   hour_cos                          55234 non-null  float64
 10  hour_sin                          55234 non-null  float64
 11  minute_cos                        55234 non

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,3.3,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,...,7.51,-2.650083,-7.15,4.43,6.569232,6.517967,6.651443,1.155989,0.978326,1.272566
2000-01-01 09:00:00,4.3,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,...,5.37,-0.691833,-4.90,4.35,6.577484,6.537126,6.650408,1.081418,1.011601,1.208960
2000-01-01 12:00:00,3.0,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,...,6.39,-0.399500,-6.13,6.58,6.619954,6.574378,6.654925,1.046978,0.896088,1.220830
2000-01-01 15:00:00,4.3,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,6.589432,6.557204,6.640529,1.133350,1.018847,1.223775
2000-01-01 18:00:00,3.7,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,...,5.81,-0.374333,-2.88,4.80,6.564192,6.527812,6.629759,1.387579,0.989541,1.658228


In [46]:
# Check for nulls after shift
kp_features_train_3hr[kp_features_train_3hr.isnull().any(axis=1)]

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2019-12-31 21:00:00,NaN,7.0,0.7,2019,365,21,0,1.0,6.432491e-16,0.707107,...,2.88,-3.3955,-4.02,-2.79,5.713915,5.67949,5.734635,2.445608,2.154085,2.565718


In [47]:
# Drop resulting nulls for train set
print(f"Records before dropping NAs: {len(kp_features_train_3hr):,}")
kp_features_train_3hr = kp_features_train_3hr.dropna(subset=['kp_index_3_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_train_3hr):,}")

Records before dropping NAs: 55,234
Records after dropping NAs: 55,233


In [48]:
kp_features_validation_3hr.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 9713 entries, 2020-01-01 06:00:00 to 2023-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   kp_index_3_hr_forecast            9712 non-null   float64
 1   kp_10                             9713 non-null   float64
 2   kp_index                          9713 non-null   float64
 3   year                              9713 non-null   int32  
 4   day                               9713 non-null   int32  
 5   hour                              9713 non-null   int32  
 6   minute                            9713 non-null   int64  
 7   day_cos                           9713 non-null   float64
 8   day_sin                           9713 non-null   float64
 9   hour_cos                          9713 non-null   float64
 10  hour_sin                          9713 non-null   float64
 11  minute_cos                        9713 non-n

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-01 06:00:00,0.7,0.0,0.0,2020,1,6,0,0.999852,0.017213,6.123234e-17,...,-0.31,1.166000,0.52,1.83,5.715800,5.700444,5.726522,2.273935,1.986504,2.534094
2020-01-01 09:00:00,0.7,7.0,0.7,2020,1,9,0,0.999852,0.017213,-7.071068e-01,...,-0.92,1.366417,0.93,2.22,5.740880,5.717028,5.749711,2.253194,2.131797,2.378620
2020-01-01 12:00:00,1.3,7.0,0.7,2020,1,12,0,0.999852,0.017213,-1.000000e+00,...,-1.43,-3.002500,-3.44,-2.58,5.799468,5.774552,5.833933,2.208855,2.085672,2.397895
2020-01-01 15:00:00,1.0,13.0,1.3,2020,1,15,0,0.999852,0.017213,-7.071068e-01,...,0.97,1.558583,-0.94,2.97,5.781606,5.771441,5.790572,2.354791,2.230014,2.483239
2020-01-01 18:00:00,0.7,10.0,1.0,2020,1,18,0,0.999852,0.017213,-1.836970e-16,...,0.86,-0.565028,-2.66,2.10,5.804606,5.766131,5.869297,2.326269,2.239645,2.392426


In [49]:
# Check for nulls after shift
kp_features_validation_3hr[kp_features_validation_3hr.isnull().any(axis=1)]

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-30 21:00:00,NaN,0.0,0.0,2023,181,21,0,-0.999667,0.025818,0.707107,...,3.11,1.705917,1.05,2.01,6.243603,6.237543,6.256901,0.973249,0.924259,1.036737


In [50]:
# Drop resulting nulls for validation set
print(f"Records before dropping NAs: {len(kp_features_validation_3hr):,}")
kp_features_validation_3hr = kp_features_validation_3hr.dropna(subset=['kp_index_3_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_validation_3hr):,}")

Records before dropping NAs: 9,713
Records after dropping NAs: 9,712


In [51]:
kp_features_test_3hr.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 8158 entries, 2023-07-01 18:00:00 to 2026-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   kp_index_3_hr_forecast            8157 non-null   float64
 1   kp_10                             8158 non-null   float64
 2   kp_index                          8158 non-null   float64
 3   year                              8158 non-null   int32  
 4   day                               8158 non-null   int32  
 5   hour                              8158 non-null   int32  
 6   minute                            8158 non-null   int64  
 7   day_cos                           8158 non-null   float64
 8   day_sin                           8158 non-null   float64
 9   hour_cos                          8158 non-null   float64
 10  hour_sin                          8158 non-null   float64
 11  minute_cos                        8158 non-n

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-07-01 18:00:00,1.3,10.0,1.0,2023,182,18,0,-0.999963,0.008607,-1.836970e-16,...,5.95,-1.277393,-1.78,-0.89,6.190540,6.179137,6.241445,1.098754,0.896088,1.141033
2023-07-01 21:00:00,0.7,13.0,1.3,2023,182,21,0,-0.999963,0.008607,7.071068e-01,...,3.46,0.402750,-0.74,1.55,6.095282,6.076495,6.110802,1.000582,0.928219,1.098612
2023-07-02 00:00:00,1.3,7.0,0.7,2023,183,0,0,-0.999963,-0.008607,1.000000e+00,...,-0.42,1.983583,1.08,2.86,6.072620,6.042870,6.104347,1.303858,1.156881,1.456287
2023-07-02 03:00:00,0.7,13.0,1.3,2023,183,3,0,-0.999963,-0.008607,7.071068e-01,...,4.75,1.654667,0.11,2.84,6.059389,6.028279,6.094472,1.508568,1.378766,1.591274
2023-07-02 06:00:00,1.3,7.0,0.7,2023,183,6,0,-0.999963,-0.008607,6.123234e-17,...,5.15,-0.022833,-2.05,0.83,6.071296,6.041920,6.082219,1.417285,1.311032,1.543298


In [52]:
# Check for nulls after shift
kp_features_test_3hr[kp_features_test_3hr.isnull().any(axis=1)]

,kp_index_3_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2026-06-30 21:00:00,NaN,33.0,3.3,2026,181,21,0,-0.999667,0.025818,0.707107,...,-4.83,-7.477333,-8.72,-6.15,6.060263,6.044768,6.077642,2.865805,2.669309,3.048325


In [53]:
# Drop resulting nulls for test set
print(f"Records before dropping NAs: {len(kp_features_test_3hr):,}")
kp_features_test_3hr = kp_features_test_3hr.dropna(subset=['kp_index_3_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_test_3hr):,}")

Records before dropping NAs: 8,158
Records after dropping NAs: 8,157


In [54]:
# Write engineered Kp interval features to parquet
train_output_path = PROCESSED_DIR / 'lr_features_3h_train.parquet'
validation_output_path = PROCESSED_DIR / 'lr_features_3h_validation.parquet'
test_output_path = PROCESSED_DIR / 'lr_features_3h_test.parquet'

kp_features_train_3hr.to_parquet(train_output_path)
kp_features_validation_3hr.to_parquet(validation_output_path)
kp_features_test_3hr.to_parquet(test_output_path)

print(f'Wrote {len(kp_features_train_3hr):,} rows to {train_output_path.as_posix()}')
print(f'Wrote {len(kp_features_validation_3hr):,} rows to {validation_output_path.as_posix()}')
print(f'Wrote {len(kp_features_test_3hr):,} rows to {test_output_path.as_posix()}')

Wrote 55,233 rows to work/Processed/lr_features_3h_train.parquet
Wrote 9,712 rows to work/Processed/lr_features_3h_validation.parquet
Wrote 8,157 rows to work/Processed/lr_features_3h_test.parquet


## 6\-hour forecast

In [55]:
# Create 6-hour forecast dataframes (train/validation/test)
kp_features_train_6hr = create_forecast_target(df=kp_features_train, forecast_horizon=6)
kp_features_validation_6hr = create_forecast_target(df=kp_features_validation, forecast_horizon=6)
kp_features_test_6hr = create_forecast_target(df=kp_features_test, forecast_horizon=6)

In [56]:
# Check train set
kp_features_train_6hr.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 55234 entries, 2000-01-01 06:00:00 to 2019-12-31 21:00:00
Data columns (total 86 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   kp_index_6_hr_forecast            55232 non-null  float64
 1   kp_10                             55234 non-null  float64
 2   kp_index                          55234 non-null  float64
 3   year                              55234 non-null  int32  
 4   day                               55234 non-null  int32  
 5   hour                              55234 non-null  int32  
 6   minute                            55234 non-null  int64  
 7   day_cos                           55234 non-null  float64
 8   day_sin                           55234 non-null  float64
 9   hour_cos                          55234 non-null  float64
 10  hour_sin                          55234 non-null  float64
 11  minute_cos                        55234 non

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 06:00:00,4.3,40.0,4.0,2000,1,6,0,0.999852,0.017213,6.123234e-17,...,7.51,-2.650083,-7.15,4.43,6.569232,6.517967,6.651443,1.155989,0.978326,1.272566
2000-01-01 09:00:00,3.0,33.0,3.3,2000,1,9,0,0.999852,0.017213,-7.071068e-01,...,5.37,-0.691833,-4.90,4.35,6.577484,6.537126,6.650408,1.081418,1.011601,1.208960
2000-01-01 12:00:00,4.3,43.0,4.3,2000,1,12,0,0.999852,0.017213,-1.000000e+00,...,6.39,-0.399500,-6.13,6.58,6.619954,6.574378,6.654925,1.046978,0.896088,1.220830
2000-01-01 15:00:00,3.7,30.0,3.0,2000,1,15,0,0.999852,0.017213,-7.071068e-01,...,5.36,1.213417,-2.47,5.16,6.589432,6.557204,6.640529,1.133350,1.018847,1.223775
2000-01-01 18:00:00,3.0,43.0,4.3,2000,1,18,0,0.999852,0.017213,-1.836970e-16,...,5.81,-0.374333,-2.88,4.80,6.564192,6.527812,6.629759,1.387579,0.989541,1.658228


In [58]:
# Check for nulls after shift
kp_features_train_6hr[kp_features_train_6hr.isnull().any(axis=1)]

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2019-12-31 18:00:00,NaN,13.0,1.3,2019,365,18,0,1.0,6.432491e-16,-1.836970e-16,...,4.66,-1.45575,-2.20,-0.49,5.714521,5.695078,5.736895,2.471319,2.333114,2.623218
2019-12-31 21:00:00,NaN,7.0,0.7,2019,365,21,0,1.0,6.432491e-16,7.071068e-01,...,2.88,-3.39550,-4.02,-2.79,5.713915,5.679490,5.734635,2.445608,2.154085,2.565718


In [59]:
# Drop resulting nulls for train set
print(f"Records before dropping NAs: {len(kp_features_train_6hr):,}")
kp_features_train_6hr = kp_features_train_6hr.dropna(subset=['kp_index_6_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_train_6hr):,}")

Records before dropping NAs: 55,234
Records after dropping NAs: 55,232


In [60]:
# Check validation set
kp_features_validation_6hr.info()
kp_features_validation_6hr.head()

<class 'pandas.DataFrame'>
DatetimeIndex: 9713 entries, 2020-01-01 06:00:00 to 2023-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   kp_index_6_hr_forecast            9711 non-null   float64
 1   kp_10                             9713 non-null   float64
 2   kp_index                          9713 non-null   float64
 3   year                              9713 non-null   int32  
 4   day                               9713 non-null   int32  
 5   hour                              9713 non-null   int32  
 6   minute                            9713 non-null   int64  
 7   day_cos                           9713 non-null   float64
 8   day_sin                           9713 non-null   float64
 9   hour_cos                          9713 non-null   float64
 10  hour_sin                          9713 non-null   float64
 11  minute_cos                        9713 non-n

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-01 06:00:00,0.7,0.0,0.0,2020,1,6,0,0.999852,0.017213,6.123234e-17,...,-0.31,1.166000,0.52,1.83,5.715800,5.700444,5.726522,2.273935,1.986504,2.534094
2020-01-01 09:00:00,1.3,7.0,0.7,2020,1,9,0,0.999852,0.017213,-7.071068e-01,...,-0.92,1.366417,0.93,2.22,5.740880,5.717028,5.749711,2.253194,2.131797,2.378620
2020-01-01 12:00:00,1.0,7.0,0.7,2020,1,12,0,0.999852,0.017213,-1.000000e+00,...,-1.43,-3.002500,-3.44,-2.58,5.799468,5.774552,5.833933,2.208855,2.085672,2.397895
2020-01-01 15:00:00,0.7,13.0,1.3,2020,1,15,0,0.999852,0.017213,-7.071068e-01,...,0.97,1.558583,-0.94,2.97,5.781606,5.771441,5.790572,2.354791,2.230014,2.483239
2020-01-01 18:00:00,0.0,10.0,1.0,2020,1,18,0,0.999852,0.017213,-1.836970e-16,...,0.86,-0.565028,-2.66,2.10,5.804606,5.766131,5.869297,2.326269,2.239645,2.392426


In [61]:
# Check for nulls after shift
kp_features_validation_6hr[kp_features_validation_6hr.isnull().any(axis=1)]

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-30 18:00:00,NaN,3.0,0.3,2023,181,18,0,-0.999667,0.025818,-1.836970e-16,...,5.00,-1.245833,-2.60,0.27,6.325264,6.258051,6.405063,1.321505,0.996949,1.566530
2023-06-30 21:00:00,NaN,0.0,0.0,2023,181,21,0,-0.999667,0.025818,7.071068e-01,...,3.11,1.705917,1.05,2.01,6.243603,6.237543,6.256901,0.973249,0.924259,1.036737


In [62]:
# Drop resulting nulls for validation set
print(f"Records before dropping NAs: {len(kp_features_validation_6hr):,}")
kp_features_validation_6hr = kp_features_validation_6hr.dropna(subset=['kp_index_6_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_validation_6hr):,}")

Records before dropping NAs: 9,713
Records after dropping NAs: 9,711


In [63]:
# Check test set
kp_features_test_6hr.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 8158 entries, 2023-07-01 18:00:00 to 2026-06-30 21:00:00
Data columns (total 86 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   kp_index_6_hr_forecast            8156 non-null   float64
 1   kp_10                             8158 non-null   float64
 2   kp_index                          8158 non-null   float64
 3   year                              8158 non-null   int32  
 4   day                               8158 non-null   int32  
 5   hour                              8158 non-null   int32  
 6   minute                            8158 non-null   int64  
 7   day_cos                           8158 non-null   float64
 8   day_sin                           8158 non-null   float64
 9   hour_cos                          8158 non-null   float64
 10  hour_sin                          8158 non-null   float64
 11  minute_cos                        8158 non-n

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2023-07-01 18:00:00,0.7,10.0,1.0,2023,182,18,0,-0.999963,0.008607,-1.836970e-16,...,5.95,-1.277393,-1.78,-0.89,6.190540,6.179137,6.241445,1.098754,0.896088,1.141033
2023-07-01 21:00:00,1.3,13.0,1.3,2023,182,21,0,-0.999963,0.008607,7.071068e-01,...,3.46,0.402750,-0.74,1.55,6.095282,6.076495,6.110802,1.000582,0.928219,1.098612
2023-07-02 00:00:00,0.7,7.0,0.7,2023,183,0,0,-0.999963,-0.008607,1.000000e+00,...,-0.42,1.983583,1.08,2.86,6.072620,6.042870,6.104347,1.303858,1.156881,1.456287
2023-07-02 03:00:00,1.3,13.0,1.3,2023,183,3,0,-0.999963,-0.008607,7.071068e-01,...,4.75,1.654667,0.11,2.84,6.059389,6.028279,6.094472,1.508568,1.378766,1.591274
2023-07-02 06:00:00,1.7,7.0,0.7,2023,183,6,0,-0.999963,-0.008607,6.123234e-17,...,5.15,-0.022833,-2.05,0.83,6.071296,6.041920,6.082219,1.417285,1.311032,1.543298


In [64]:
# Check for nulls after shift
kp_features_test_6hr[kp_features_test_6hr.isnull().any(axis=1)]

,kp_index_6_hr_forecast,kp_10,kp_index,year,day,hour,minute,day_cos,day_sin,hour_cos,...,by_gsm_nt_max_3_4h,bz_gsm_nt_avg_3_4h,bz_gsm_nt_min_3_4h,bz_gsm_nt_max_3_4h,flow_speed_km_s_log_avg_3_4h,flow_speed_km_s_log_min_3_4h,flow_speed_km_s_log_max_3_4h,proton_density_n_cc_log_avg_3_4h,proton_density_n_cc_log_min_3_4h,proton_density_n_cc_log_max_3_4h
datetime,,,,,,,,,,,,,,,,,,,,,
2026-06-30 18:00:00,NaN,47.0,4.7,2026,181,18,0,-0.999667,0.025818,-1.836970e-16,...,-7.14,-0.802500,-6.45,7.81,6.076919,6.059590,6.096950,2.554655,2.346602,2.675527
2026-06-30 21:00:00,NaN,33.0,3.3,2026,181,21,0,-0.999667,0.025818,7.071068e-01,...,-4.83,-7.477333,-8.72,-6.15,6.060263,6.044768,6.077642,2.865805,2.669309,3.048325


In [65]:
# Drop resulting nulls for test set
print(f"Records before dropping NAs: {len(kp_features_test_6hr):,}")
kp_features_test_6hr = kp_features_test_6hr.dropna(subset=['kp_index_6_hr_forecast'])
print(f"Records after dropping NAs: {len(kp_features_test_6hr):,}")

Records before dropping NAs: 8,158
Records after dropping NAs: 8,156


In [66]:
# Write engineered Kp interval features to parquet
train_output_path = PROCESSED_DIR / 'lr_features_6h_train.parquet'
validation_output_path = PROCESSED_DIR / 'lr_features_6h_validation.parquet'
test_output_path = PROCESSED_DIR / 'lr_features_6h_test.parquet'

kp_features_train_6hr.to_parquet(train_output_path)
kp_features_validation_6hr.to_parquet(validation_output_path)
kp_features_test_6hr.to_parquet(test_output_path)

print(f'Wrote {len(kp_features_train_6hr):,} rows to {train_output_path.as_posix()}')
print(f'Wrote {len(kp_features_validation_6hr):,} rows to {validation_output_path.as_posix()}')
print(f'Wrote {len(kp_features_test_6hr):,} rows to {test_output_path.as_posix()}')

Wrote 55,232 rows to work/Processed/lr_features_6h_train.parquet
Wrote 9,711 rows to work/Processed/lr_features_6h_validation.parquet
Wrote 8,156 rows to work/Processed/lr_features_6h_test.parquet


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>